# 12 — Lalu lintas dan mobilitas (data sintetis)

Data perjalanan ojol dan kecepatan lalu lintas **tidak terbuka**: milik perusahaan aplikasi dan penyedia
peta. Supaya alur kerjanya bisa dilatih, notebook ini **membangkitkan data sintetis** yang masuk akal dari
data terbuka kota kita — lalu menganalisisnya persis seperti data sungguhan. Semua angka di sini adalah
hasil simulasi, **bukan** pola mobilitas Semarang yang sebenarnya.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import geopandas as gpd
import h3
import numpy as np
import pandas as pd

from kota import CFG, OSM, PROC

rng = np.random.default_rng(42)
bangunan = gpd.read_file(PROC / "bangunan_utm.gpkg", layer="bangunan", columns=["tapak_m2"])
poi = gpd.read_file(OSM, layer="poi")

## 1. Model pembangkit: asal dan tujuan per heksagon H3
- **Asal** (penjemputan) sebanding dengan luas tapak bangunan di sel — proksi jumlah penghuni.
- **Tujuan** sebanding dengan jumlah POI (sekolah, pasar, fasilitas kesehatan, toko) ditambah sebagian
  luas tapak — tempat orang bekerja dan berbelanja.
- Pilihan tujuan mengikuti **model gravitasi**: makin jauh, makin kecil peluangnya (peluruhan 3 km).

In [ ]:
RES = 8
pt = bangunan.representative_point().to_crs(4326)
bangunan["h3"] = [h3.latlng_to_cell(y, x, RES) for x, y in zip(pt.x, pt.y)]
poi["h3"] = [h3.latlng_to_cell(y, x, RES) for x, y in zip(poi.geometry.x, poi.geometry.y)]

sel = pd.DataFrame({"tapak": bangunan.groupby("h3")["tapak_m2"].sum()})
sel["poi"] = poi.groupby("h3").size()
sel = sel.fillna(0)
sel["bangkitan"] = sel["tapak"] / sel["tapak"].sum()
tarik = sel["poi"] * 5000 + sel["tapak"] * 0.3
sel["tarikan"] = tarik / tarik.sum()
latlng = np.array([h3.cell_to_latlng(c) for c in sel.index])
print(len(sel), "sel H3 res", RES)

In [ ]:
N = 50_000
o_idx = rng.choice(len(sel), size=N, p=sel["bangkitan"].values)
# jarak antar-sel (km, pendekatan ekuirektangular — cukup untuk skala kota)
lat, lon = np.radians(latlng[:, 0]), np.radians(latlng[:, 1])
d_idx = np.empty(N, dtype=int)
for o in np.unique(o_idx):
    m = o_idx == o
    dx = (lon - lon[o]) * np.cos(lat[o]) * 6371
    dy = (lat - lat[o]) * 6371
    w = sel["tarikan"].values * np.exp(-np.hypot(dx, dy) / 3.0)
    w[o] *= 0.2  # perjalanan di dalam sel yang sama jarang memakai ojol
    d_idx[m] = rng.choice(len(sel), size=m.sum(), p=w / w.sum())

## 2. Waktu perjalanan
Dua puncak: pagi (07.00) dan sore (17.00), plus puncak kecil makan siang.

In [ ]:
jam_bobot = np.array([1, 1, 1, 1, 2, 5, 12, 18, 12, 8, 7, 8, 10, 8, 7, 9, 13, 17, 12, 8, 6, 4, 2, 1], float)
jam = rng.choice(24, size=N, p=jam_bobot / jam_bobot.sum())
trips = pd.DataFrame({"jam": jam, "asal_h3": sel.index[o_idx], "tujuan_h3": sel.index[d_idx]})
trips["jarak_km"] = [h3.great_circle_distance(h3.cell_to_latlng(a), h3.cell_to_latlng(b), unit="km") * 1.3
                     for a, b in zip(trips["asal_h3"], trips["tujuan_h3"])]  # ×1,3 ≈ circuity jaringan
trips["sintetis"] = True
trips["jarak_km"].describe().round(2)

## 3. Heatmap penjemputan per jam

In [ ]:
pagi = trips[trips["jam"].between(6, 8)].groupby("asal_h3").size().rename("jemput_pagi")
sore = trips[trips["jam"].between(16, 18)].groupby("tujuan_h3").size().rename("antar_sore")
hm = pd.concat([pagi, sore], axis=1).fillna(0)
from shapely.geometry import Polygon
hm = gpd.GeoDataFrame(hm, geometry=[Polygon([(x, y) for y, x in h3.cell_to_boundary(c)]) for c in hm.index],
                      crs=4326)
print("sel tersibuk pagi:", hm["jemput_pagi"].idxmax(), int(hm["jemput_pagi"].max()), "penjemputan")
ax = hm.plot(column="jemput_pagi", cmap="inferno_r", figsize=(9, 7), legend=True)
ax.set_axis_off()
ax.set_title("Penjemputan ojol 06.00–08.59 (SINTETIS)")

## 4. Kecepatan lalu lintas sintetis
Kecepatan per ruas = kecepatan bebas menurut kelas jalan × (1 − kemacetan). Kemacetan membesar pada jam
sibuk dan pada ruas ber-*betweenness* tinggi (notebook 08) — ruas yang banyak dilalui jalur terpendek.

In [ ]:
ruas = gpd.read_file(PROC / "betweenness.gpkg", layer="ruas")
hw = ruas["highway"].astype(str)
ruas = ruas[hw.str.contains("primary|secondary|tertiary|trunk")].copy()
bebas = {"trunk": 60, "primary": 50, "secondary": 40, "tertiary": 30}
ruas["v_bebas"] = [next((v for k, v in bebas.items() if k in h), 30) for h in ruas["highway"].astype(str)]
b = ruas["betweenness"].rank(pct=True)
puncak = (jam_bobot / jam_bobot.max())
rows = []
for j in range(24):
    macet = np.clip(0.65 * puncak[j] * b + rng.normal(0, 0.05, len(ruas)), 0, 0.85)
    rows.append(pd.DataFrame({"u": ruas["u"], "v": ruas["v"], "key": ruas["key"], "jam": j,
                              "kecepatan_kmj": (ruas["v_bebas"] * (1 - macet)).round(1)}))
lalin = pd.concat(rows, ignore_index=True)
lalin["sintetis"] = True
lalin.groupby("jam")["kecepatan_kmj"].median().round(1).to_frame().T

In [ ]:
trips.to_csv(PROC / "ojol_sintetis.csv", index=False)
lalin.to_csv(PROC / "lalin_sintetis.csv", index=False)
ruas[["u", "v", "key", "nama", "highway", "v_bebas", "geometry"]].to_file(PROC / "ruas_utama.gpkg", layer="ruas")
hm.to_file(PROC / "ojol_h3.gpkg", layer="h3")
print("disimpan: ojol_sintetis.csv, lalin_sintetis.csv, ruas_utama.gpkg, ojol_h3.gpkg (SEMUA SINTETIS)")